# Notebook 08 - Expansion-Aware Autorouting

This final course notebook handles hot-line routing where geometry alone is not enough. It compares direct grid candidates with generated U-loop candidates, exports Code_Aster study folders, and visualizes the reserved expansion-loop envelope.

Important boundary: with `RUN_CODE_ASTER = False`, this notebook exports solver studies and route metadata only. Enable solver execution only when Code_Aster is configured, then review imported result artifacts before treating stress, displacement, or reactions as engineering results.


## Review Goal

The primary question is visible in the interactive scene: did the selected hot-line route reserve enough space for thermal movement while avoiding equipment and future nearby lines?

The scene should show selected route, alternate candidates, obstacles, endpoints, and the reserved expansion-loop envelope.


## 1. Setup


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "tuba").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT


## 2. Build the Hot-Line Model and Request


In [ ]:
from tuba import Model
from tuba.analysis.code_aster_notebook import configure_code_aster_notebook_runtime
from tuba.routing import (
    AutoroutingAgent,
    ExpansionAwareRouter,
    ExpansionLoopGenerator,
    ExpansionLoopSpec,
    GridRouter,
    SolverAcceptanceCriteria,
    ThermalRouteRequirement,
)
from tuba.routing.solver_loop import SolverLoopConfig
from tuba.routing.types import PipeRouteRequest, RouteEndpoint, RoutingConstraints, RoutingGridSpec
from tuba.visualizer.notebook import configure_notebook_backend

JUPYTER_BACKEND = configure_notebook_backend()


def build_hot_line_model() -> Model:
    model = Model("NotebookHotLineExpansionLoop")
    model.add_material(
        "steel",
        E=210e9,
        nu=0.3,
        rho=7850.0,
        alpha=12e-6,
        allowable_stress={20.0: 140e6, 180.0: 120e6},
    )
    model.add_pipe_section("DN80", OD=0.0889, WT=0.00549)
    model.define_load_case("Hot", gravity=True, pressure=1.2e6, temperature=180.0)
    model.add_obstacle(
        id="hot_equipment",
        type="cuboid",
        min_point=(3.4, -0.35, -0.35),
        max_point=(4.6, 0.35, 0.35),
    )
    return model


def build_hot_line_request() -> PipeRouteRequest:
    return PipeRouteRequest(
        id="HOT-NB-100",
        start=RouteEndpoint("PumpDischarge", (0.0, 0.0, 0.0), direction=(1.0, 0.0, 0.0)),
        goal=RouteEndpoint("RackTieIn", (8.0, 0.0, 0.0), direction=(1.0, 0.0, 0.0)),
        section="DN80",
        material="steel",
        constraints=RoutingConstraints(
            clearance=0.10,
            insulation_thickness=0.05,
            min_bend_radius=0.25,
        ),
        thermal_requirements=ThermalRouteRequirement(
            design_temperature_c=180.0,
            reference_temperature_c=20.0,
            line_length_m=8.0,
            thermal_expansion_coefficient=12e-6,
            metadata={"service": "hot oil"},
        ),
        solver_acceptance=SolverAcceptanceCriteria.hot_line_defaults(),
    )


model = build_hot_line_model()
request = build_hot_line_request()
request


## 3. Generate Expansion-Aware Candidates


In [ ]:
router = ExpansionAwareRouter(
    base_router=GridRouter(
        RoutingGridSpec(cell_size=0.5, margin=1.5),
        candidate_count=1,
    ),
    loop_generator=ExpansionLoopGenerator(
        loop_specs=(
            ExpansionLoopSpec(
                family="u_loop",
                width_m=2.0,
                depth_m=0.8,
                plane="xy",
                min_clearance_m=0.15,
            ),
        ),
    ),
)

CODE_ASTER_RUNTIME = configure_code_aster_notebook_runtime()
RUN_CODE_ASTER = False

agent = AutoroutingAgent(
    router=router,
    solver_config=SolverLoopConfig(
        run_solver=RUN_CODE_ASTER,
        export_study=True,
        exec_method=CODE_ASTER_RUNTIME.exec_method,
        wsl_distro=CODE_ASTER_RUNTIME.wsl_distro,
        docker_image=CODE_ASTER_RUNTIME.docker_image,
        max_solver_candidates=2,
        load_case="Hot",
    ),
    output_root=PROJECT_ROOT / "routing_reports",
)

run = agent.route_pipe(model, request, apply=False)
selected = run.result.selected
selected.metadata if selected is not None else None

## 4. Compare Candidates


In [ ]:
candidate_rows = []
for idx, candidate in enumerate(run.result.candidates):
    candidate_rows.append(
        {
            "index": idx,
            "family": candidate.metadata.get("route_family", "unknown"),
            "valid": candidate.is_valid,
            "cost": round(candidate.cost, 3),
            "length": round(candidate.cost_breakdown.get("length", 0.0), 3),
            "bends": candidate.cost_breakdown.get("bends", 0.0),
            "has_reserved_envelope": "reserved_envelope" in candidate.metadata,
            "diagnostics": "; ".join(candidate.diagnostics),
        }
    )

candidate_rows

In [ ]:
print(f"Selected family: {selected.metadata.get('route_family') if selected else None}")
print(f"Report: {run.report_path}")
print(f"Study directory: {selected.metadata.get('solver', {}).get('study_dir') if selected else None}")

## Interactive 3D Review

The next cell exports a standalone HTML scene and opens the same route scene in the notebook. The transparent reserved envelope is the space that later network routes should avoid.


In [ ]:
from IPython.display import Markdown, display
from tuba.routing.visualization import export_route_scene_html, show_route_scene

scene_path = PROJECT_ROOT / "routing_reports" / request.id / "route_scene.html"

try:
    exported_scene = export_route_scene_html(model, scene_path, request=request, result=run.result)
    display(Markdown(f"Standalone scene: `{exported_scene}`"))
    show_route_scene(model, request=request, result=run.result, jupyter_backend=JUPYTER_BACKEND)
except ImportError as exc:
    display(Markdown(f"PyVista notebook visualization is not installed: `{exc}`"))
except Exception as exc:
    display(Markdown(f"Scene export/display failed in this environment: `{exc}`"))


## What To Review

- Does the selected U-loop reserve enough volume around the offset leg?
- Is the grid detour cheaper only because it ignores thermal flexibility, or is the loop actually lower cost?
- Do exported Code_Aster study files exist for the reviewed candidates?
- If solver execution is enabled later, do expansion ratio, sustained ratio, and anchor reaction stay inside the project limits?
- Do nearby network routes intersect the selected route's reserved envelope?